# 05 · Multi-layer batch processing

Real datasets are usually 3D — multiple Z-layers each with their own
stack of detector frames. The pipeline batches them via
`startLayerNr..endLayerNr`:

- Each layer gets its own `<resultFolder>/LayerNr_<n>/` workspace.
- The param file's `RawStartNr` is offset per layer:
    `RawStartNr_layer_n = RawStartNr_original + (n-1) * nDistances * NrFilesPerDistance`
- Per-layer `OutputDirectory` is forced to `<resultFolder>/LayerNr_<n>/`.
- After each layer, `Mic2GrainsList` runs once on the final mic →
  `<resultFolder>/GrainsLayer<n>.csv` (the seed source for the next experiment).

In [ ]:
import os, shutil, tempfile
from pathlib import Path
from argparse import Namespace

from midas_nf_pipeline.workflows import run_multi_layer

In [ ]:
MIDAS_HOME = Path(os.environ.get('MIDAS_HOME') or '.')
AU_DIR = MIDAS_HOME / 'NF_HEDM/Example/sim'
ws = Path(tempfile.mkdtemp(prefix='nf_multilayer_'))
param = ws / 'test_ps_au.txt'
shutil.copy2(AU_DIR / 'test_ps_au.txt', param)
with open(param, 'a') as f:
    f.write(f'OutputDirectory {ws}\n')
    f.write('NrFilesPerDistance 1\n')
print('workspace:', ws)

In [ ]:
# Run 3 layers (1, 2, 3). The Au example only has data for layer 1, so
# layers 2/3 will fail unless your dataset has >1 layer — replace `param`
# with your real param file.
args = Namespace(
    paramFN=str(param), nCPUs=4, device='auto',
    ffSeedOrientations=False, doImageProcessing=1,
    startLayerNr=1, endLayerNr=1,             # bump to 3 for a real dataset
    resultFolder=str(ws), minConfidence=0.6,
    resume='', restartFrom='',
)
h5_paths = run_multi_layer(args, install_dir=str(MIDAS_HOME))
for p in h5_paths:
    print('layer H5:', p)

In [ ]:
# After all layers complete, you'll find:
#   <ws>/LayerNr_1/  ... LayerNr_N/      — per-layer outputs
#   <ws>/GrainsLayer1.csv ... GrainsLayerN.csv
for p in sorted(ws.glob('GrainsLayer*.csv')):
    head = p.read_text().splitlines()[:3]
    print(p.name, head)

## CLI version

```bash
midas-nf-pipeline run params.txt \
    --start-layer 1 --end-layer 25 \
    --result-folder /scratch/my_dataset \
    --n-cpus 16 --device cuda
```